# Vendor Sentiment Analysis
Keyword-based NLP — Classify review sentiment and track weekly trends.

Run cells top-to-bottom: **Setup → Generate Data → Train → Evaluate → Test**

## 1. Setup
Resolve paths for `data/`, `output/`, `report/` directories.

In [ ]:
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()

BASE_DIR   = NOTEBOOK_DIR.parent
DATA_DIR   = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
REPORT_DIR = BASE_DIR / "report"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR   : {BASE_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"REPORT_DIR : {REPORT_DIR}")

## 2. Generate Data
Create synthetic training data and save to `data/`.

In [ ]:
import csv
import random
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
DATA_DIR = SCRIPT_DIR.parent / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)

items = ['satay', 'bubble tea', 'nasi lemak', 'murtabak', 'ice cream',
         'fried chicken', 'kueh', 'rojak']
quality_pos = ['fresh', 'tender', 'delicious', 'crispy', 'juicy']
quality_neg = ['stale', 'tough', 'cold', 'charred', 'soggy']
emotions = ['satisfied', 'happy', 'pleased']

pos_templates = [
    "Amazing [item]! Best I've had.",
    "Love the [item] here, always [quality].",
    "Great value, highly recommend the [item].",
    "The [item] was [quality], so [emotion]!",
    "Perfectly [quality] [item], will come back."
]

neg_templates = [
    "Disappointing [item], expected better.",
    "Too long a queue, [item] was lukewarm.",
    "Overpriced for the portion size.",
    "The [item] was tough and cold.",
    "Poor quality [item], not worth the wait."
]

# Weekly positive rates
weekly_pos_rates = {1: 0.575, 2: 0.545, 3: 0.630, 4: 0.560}

rows = []
review_id = 1

for week in range(1, 5):
    pos_rate = weekly_pos_rates[week]
    n_pos = round(200 * pos_rate)
    n_neg = 200 - n_pos

    week_reviews = []
    for _ in range(n_pos):
        item = random.choice(items)
        template = random.choice(pos_templates)
        quality = random.choice(quality_pos)
        emotion = random.choice(emotions)
        text = template.replace("[item]", item).replace("[quality]", quality).replace("[emotion]", emotion)
        rating = random.randint(4, 5)
        week_reviews.append({"text": text, "label": "POSITIVE", "rating": rating})

    for _ in range(n_neg):
        item = random.choice(items)
        template = random.choice(neg_templates)
        quality = random.choice(quality_neg)
        emotion = random.choice(emotions)
        text = template.replace("[item]", item).replace("[quality]", quality).replace("[emotion]", emotion)
        rating = random.randint(1, 2)
        week_reviews.append({"text": text, "label": "NEGATIVE", "rating": rating})

    random.shuffle(week_reviews)

    for rev in week_reviews:
        rows.append({
            "id": review_id,
            "week": week,
            "text": rev["text"],
            "true_label": rev["label"],
            "rating": rev["rating"]
        })
        review_id += 1

out_path = DATA_DIR / "reviews_data.csv"
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "week", "text", "true_label", "rating"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} rows to {out_path}")

## 3. Train
Fit the model and save to `output/`.

In [ ]:
import pickle
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
OUTPUT_DIR = SCRIPT_DIR.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pos_keywords = [
    'amazing', 'great', 'love', 'best', 'fresh', 'tender', 'delicious',
    'fantastic', 'perfect', 'recommend', 'awesome', 'tasty', 'good',
    'crispy', 'juicy', 'satisfied', 'happy'
]

neg_keywords = [
    'disappointing', 'overpriced', 'tough', 'cold', 'long', 'wait',
    'bad', 'queue', 'lukewarm', 'charred', 'poor', 'stale', 'soggy',
    'worst', 'terrible', 'slow'
]

model = {
    'pos_keywords': pos_keywords,
    'neg_keywords': neg_keywords,
    'threshold': 0
}

out_path = OUTPUT_DIR / "sentiment_model.pkl"
with open(out_path, "wb") as f:
    pickle.dump(model, f)

print(f"Model saved to {out_path}")
print(f"Positive keywords ({len(pos_keywords)}): {pos_keywords}")
print(f"Negative keywords ({len(neg_keywords)}): {neg_keywords}")

## 4. Evaluate
Compute metrics on the test set and save to `report/`.

In [ ]:
import csv
import pickle
import json
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
OUTPUT_DIR = SCRIPT_DIR.parent / "output"
DATA_DIR = SCRIPT_DIR.parent / "data"
REPORT_DIR = SCRIPT_DIR.parent / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def classify(text, pos_kw, neg_kw, threshold=0):
    text_lower = text.lower()
    score = (sum(1 for kw in pos_kw if kw in text_lower)
             - sum(1 for kw in neg_kw if kw in text_lower))
    return "POSITIVE" if score >= threshold else "NEGATIVE"

with open(OUTPUT_DIR / "sentiment_model.pkl", "rb") as f:
    model = pickle.load(f)

pos_kw = model["pos_keywords"]
neg_kw = model["neg_keywords"]
threshold = model["threshold"]

rows = []
with open(DATA_DIR / "reviews_data.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

y_true = [r["true_label"] for r in rows]
y_pred = [classify(r["text"], pos_kw, neg_kw, threshold) for r in rows]

# Overall metrics
tp = sum(1 for t, p in zip(y_true, y_pred) if t == "POSITIVE" and p == "POSITIVE")
tn = sum(1 for t, p in zip(y_true, y_pred) if t == "NEGATIVE" and p == "NEGATIVE")
fp = sum(1 for t, p in zip(y_true, y_pred) if t == "NEGATIVE" and p == "POSITIVE")
fn = sum(1 for t, p in zip(y_true, y_pred) if t == "POSITIVE" and p == "NEGATIVE")

accuracy = round((tp + tn) / len(y_true), 4)
precision = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0.0
recall = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0.0
f1 = round(2 * precision * recall / (precision + recall), 4) if (precision + recall) > 0 else 0.0

# Weekly trend
weekly_trend = []
for week in range(1, 5):
    week_rows = [(t, p) for r, t, p in zip(rows, y_true, y_pred) if int(r["week"]) == week]
    count = len(week_rows)
    pos_pct = round(sum(1 for _, p in week_rows if p == "POSITIVE") / count * 100, 1)
    neg_pct = round(100 - pos_pct, 1)
    weekly_trend.append({
        "week": f"Week {week}",
        "positive": pos_pct,
        "negative": neg_pct,
        "reviews": count
    })

overall_pos_pct = round(sum(1 for p in y_pred if p == "POSITIVE") / len(y_pred) * 100, 1)

report = {
    "model": "Keyword NLP",
    "task": "Review sentiment classification",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "weekly_trend": weekly_trend,
    "overall_positive_pct": overall_pos_pct
}

out_path = REPORT_DIR / "evaluation_report.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Report saved to {out_path}")
print(f"Accuracy={accuracy}  Precision={precision}  Recall={recall}  F1={f1}")
print(f"Overall positive: {overall_pos_pct}%")

## 5. Test
Run inference on sample inputs.

In [ ]:
import pickle
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
OUTPUT_DIR = SCRIPT_DIR.parent / "output"

def classify(text, pos_kw, neg_kw, threshold=0):
    text_lower = text.lower()
    score = (sum(1 for kw in pos_kw if kw in text_lower)
             - sum(1 for kw in neg_kw if kw in text_lower))
    return "POSITIVE" if score >= threshold else "NEGATIVE", score

with open(OUTPUT_DIR / "sentiment_model.pkl", "rb") as f:
    model = pickle.load(f)

pos_kw = model["pos_keywords"]
neg_kw = model["neg_keywords"]
threshold = model["threshold"]

sample_reviews = [
    ("Amazing satay! Best I've had.", "POSITIVE"),
    ("Disappointing nasi lemak, expected better.", "NEGATIVE"),
    ("Love the bubble tea here, always fresh.", "POSITIVE"),
    ("The murtabak was tough and cold.", "NEGATIVE"),
    ("Great value, highly recommend the kueh.", "POSITIVE"),
]

print("\n=== Vendor Sentiment Analysis: Sample Review Classifications ===")
print(f"{'Text':<50} {'Expected':<10} {'Predicted':<10} {'Score':>5}")
for text, expected in sample_reviews:
    pred, score = classify(text, pos_kw, neg_kw, threshold)
    match = "OK" if pred == expected else "FAIL"
    print(f"{text[:48]:<50} {expected:<10} {pred:<10} {score:>5}  {match}")